In [4]:
import pandas as pd
import numpy as np
import heapq
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv('df_analyse_amont.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'df_analyse_amont.csv'

In [3]:
chemin = 'BD_2016_2020_TableauDynamique.xlsx'
df_original = pd.read_excel(chemin, sheet_name="Extraction juin 2016 juin 2020")

In [12]:
df_copy = df_original.copy()

In [8]:
df_original.columns

Index(['Num d'ordre', 'Date', 'Heure',
       ' Informations du passage->date et heure d'arrivée',
       ' Informations du passage->moyen d'arrivée', ' IAO->motif d'entrée',
       ' IAO->motif de venue', ' Patient->age en année',
       ' Antécédent->antécédents médicaux',
       ' Antécédent->antécédents chirurgicaux',
       ' Constante->Surveillances->PAS/PAD adultes (première valeur)',
       ' Constante->Surveillances->FC adultes (première valeur)',
       ' Constante->Surveillances->Température adultes (première valeur)',
       ' Constante->Surveillances->SaO2 adultes (première valeur)',
       ' Constante->Surveillances->Fréquence Respiratoire (première valeur)',
       ' Diagnostic->Code CCMU', ' Localisation->Salles',
       ' Localisation->date d'entrée en box', ' Anamnèse->texte',
       ' IAO->observation',
       ' Examens complémentaires->a des examens de biologie',
       ' Examens complémentaires->a des examens de radiologie',
       ' Examens complémentaires->a des 

In [10]:
df_copy[" Informations du passage->date début prise en charge médicale"]

0         42522.043750
1         42522.093056
2                  NaN
3         42522.081250
4         42522.107639
              ...     
336248    43984.075694
336249             NaN
336250    43984.128472
336251    43984.140972
336252             NaN
Name:  Informations du passage->date début prise en charge médicale, Length: 336253, dtype: float64

In [11]:
cols = [
    " Informations du passage->date début prise en charge médicale",
    " Examens complémentaires->date et heure de première prescription de biologie"
]

for col in cols:
    df_copy[col] = pd.to_datetime(
        df_copy[col],
        unit="d",
        origin="1899-12-30",
        errors="coerce"
    )

In [ ]:
df_copy[[
    " Informations du passage->date début prise en charge médicale",
    " Examens complémentaires->date et heure de première prescription de biologie",
]].dropna().head(15)

,Informations du passage->date début prise en charge médicale,Examens complémentaires->date et heure de première prescription de biologie
0,2016-06-01 01:02:59.999999749,2016-06-01 01:30:00.000000000
1,2016-06-01 02:13:59.999995407,2016-06-01 01:34:00.000002040
3,2016-06-01 01:57:00.000000250,2016-06-01 01:39:59.999996298
4,2016-06-01 02:35:00.000000769,2016-06-01 01:34:00.000002040
6,2016-06-01 02:17:00.000001020,2016-06-01 02:17:00.000001020
7,2016-06-01 02:30:59.999998728,2016-06-01 02:25:59.999992095
8,2016-06-01 02:32:59.999999749,2016-06-01 02:32:59.999999749
10,2016-06-01 02:57:00.000003182,2016-06-01 05:41:00.000003831
11,2016-06-01 02:58:59.999995407,2016-06-01 02:58:59.999995407
12,2016-06-01 04:18:59.999999109,2016-06-01 04:18:59.999999109


In [ ]:
df_copy["temps_passe_box_min"] = (
    df_copy[" Informations du passage->date début prise en charge médicale"]
    - df_copy[" Examens complémentaires->date et heure de première prescription de biologie"]
).dt.total_seconds() / 60

In [ ]:
df_copy[[
    " Examens complémentaires->date et heure de première prescription de biologie",
    " Orientation->type d'orientation"
]].dropna().head(15)

0    42522.062500
1    42522.065278
3    42522.069444
4    42522.065278
6    42522.095139
Name:  Examens complémentaires->date et heure de première prescription de biologie, dtype: float64

In [ ]:
df_copy["temps_passe_box_min"].head(20)

0     -27.0
1      40.0
2       NaN
3      17.0
4      61.0
5       NaN
6       0.0
7       5.0
8       0.0
9       NaN
10   -164.0
11      0.0
12      0.0
13     18.0
14      NaN
15      NaN
16      NaN
17     38.0
18     -3.0
19     28.0
Name: temps_passe_box_min, dtype: float64

In [ ]:
df_original.head()

In [ ]:
df.head()

,num_dordre,date,heure,_informations_du_passage_date_et_heure_darrive,_informations_du_passage_moyen_darrive,_iao_motif_dentre,_iao_motif_de_venue,_patient_age_en_anne,_antcdent_antcdents_mdicaux,_antcdent_antcdents_chirurgicaux,...,is_retour_domicile,is_transfert,is_hospit,delai_arrivee_PEC_cap,delai_arrivee_box_cap,ind_complexite,ind_chronicite_delais,ind_issue,ind_congestion,ind_specialisation_examens
0,1,2016-06-01,00:16:00,2016-06-01 00:16:00,AMBULANCE PRIVEE,"Malaise, difficulté respi, palpitation, Pas de...",Cardiologie: Palpitations Malaise,33,NaN,NaN,...,1,0,0,47.0,50.0,0.118129,-1.090800,1.308571,-1.855631,-0.624738
1,2,2016-06-01,00:18:00,2016-06-01 00:18:00,AMBULANCE PRIVEE,"AEG, Hyperthermie et difficulté respi en maiso...",Maladies infectieuses: Hyperthermie,78,NaN,NaN,...,0,1,0,116.0,74.0,2.192266,0.107738,-1.414844,-1.254767,-1.074449
2,3,2016-06-01,00:36:00,2016-06-01 00:36:00,POMPIERS,NaN,NaN,77,NaN,NaN,...,0,1,0,NaN,5.0,0.096306,-0.465269,-2.492932,-1.216424,1.195746
3,4,2016-06-01,00:43:00,2016-06-01 00:43:00,MOYENS PERSONNELS,"1A, Hyperthermie ne cédant pas au paracetamol,...",Maladies infectieuses: Hyperthermie,38,NaN,NaN,...,1,0,0,74.0,53.0,-1.001173,0.115672,0.395099,-0.513320,-0.695704
4,5,2016-06-01,01:16:00,2016-06-01 01:16:00,AMBULANCE PRIVEE,"Douleur bras gauche et mollet gauche, hémiplég...",Cardiologie: Phlébite,69,décembre 2007 : infarctus capsulo caudé droit ...,NaN,...,1,0,0,79.0,20.0,-0.000047,0.384547,0.760444,-1.609653,1.700027


In [ ]:
df["_informations_du_passage_date_dbut_prise_en_charge_mdicale_dt"].head()

0    2016-06-01 01:02:59.999999749
1    2016-06-01 02:13:59.999995407
2                              NaN
3    2016-06-01 01:57:00.000000250
4    2016-06-01 02:35:00.000000769
Name: _informations_du_passage_date_dbut_prise_en_charge_mdicale_dt, dtype: object

In [ ]:
df["datetime_arrivee"].head()

0    2016-06-01 00:16:00
1    2016-06-01 00:18:00
2    2016-06-01 00:36:00
3    2016-06-01 00:43:00
4    2016-06-01 01:16:00
Name: datetime_arrivee, dtype: object

In [ ]:
df["datetime_sortie"].head()

0    2016-06-01 04:14:00.000001271
1    2016-06-01 11:05:00.000003701
2    2016-06-01 03:04:00.000002040
3    2016-06-01 04:50:00.000000769
4    2016-06-01 15:54:59.999996298
Name: datetime_sortie, dtype: object

In [ ]:
df["delai_total_calc"].head()

0    238.0
1    647.0
2    148.0
3    247.0
4    879.0
Name: delai_total_calc, dtype: float64

In [ ]:
df["_localisation_date_dentre_en_box_dt"].head()

0    2016-06-01 01:05:59.999997189
1    2016-06-01 01:32:00.000001020
2    2016-06-01 00:40:59.999997959
3    2016-06-01 01:36:00.000003061
4    2016-06-01 01:36:00.000003061
Name: _localisation_date_dentre_en_box_dt, dtype: object

In [ ]:
df["delai_arrivee_box_cap"].head()

0    50.0
1    74.0
2     5.0
3    53.0
4    20.0
Name: delai_arrivee_box_cap, dtype: float64

In [ ]:
df.columns

Index(['num_dordre', 'date', 'heure',
       '_informations_du_passage_date_et_heure_darrive',
       '_informations_du_passage_moyen_darrive', '_iao_motif_dentre',
       '_iao_motif_de_venue', '_patient_age_en_anne',
       '_antcdent_antcdents_mdicaux', '_antcdent_antcdents_chirurgicaux',
       '_constante_surveillances_paspad_adultes_premire_valeur',
       '_constante_surveillances_fc_adultes_premire_valeur',
       '_constante_surveillances_temprature_adultes_premire_valeur',
       '_constante_surveillances_sao2_adultes_premire_valeur',
       '_constante_surveillances_frquence_respiratoire_premire_valeur',
       '_diagnostic_code_ccmu', '_localisation_salles',
       '_localisation_date_dentre_en_box', '_anamnse_texte',
       '_iao_observation', '_examens_complmentaires_a_des_examens_de_biologie',
       '_examens_complmentaires_a_des_examens_de_radiologie',
       '_examens_complmentaires_a_des_examens_dchographie',
       '_examens_complmentaires_a_des_examens_de_scanner',

In [ ]:
(df["_informations_du_passage_date_et_heure_darrive"] == df["datetime_arrivee"]).value_counts()

True    336253
Name: count, dtype: int64

In [ ]:
df["datetime_arrivee"].head()

0    2016-06-01 00:16:00
1    2016-06-01 00:18:00
2    2016-06-01 00:36:00
3    2016-06-01 00:43:00
4    2016-06-01 01:16:00
Name: datetime_arrivee, dtype: object

In [ ]:
df['delai_arrivee_box'].head()

0    50.0
1    74.0
2     5.0
3    53.0
4    20.0
Name: delai_arrivee_box, dtype: float64

In [ ]:
df["delai_arrivee_PEC_cap"].head(15)

0      47.0
1     116.0
2       NaN
3      74.0
4      79.0
5     100.0
6      47.0
7      48.0
8      48.0
9      31.0
10     33.0
11     32.0
12     50.0
13     55.0
14     18.0
Name: delai_arrivee_PEC_cap, dtype: float64

### FIFO (First In First Out)

* datetime_arrivee : TimeStamp de l'arrivée aux urgences
* delai_arrivee_box_cap : tamps d'attente avant d'entrer dans le box (_localisation_date_dentre_en_box_dt - datetime_arrivee)
* _localisation_date_dentre_en_box_dt : TimeStamp de l'entrée dans le box
* _informations_du_passage_date_dbut_prise_en_charge_mdicale_dt : TimeStamp de la PEC
* IL FAUT CRéER  temps passé dans le box = _informations_du_passage_date_dbut_prise_en_charge_mdicale_dt - _localisation_date_dentre_en_box_dt
* datetime_sortie : TimeStamp de la sortie aux urgences
* delai_total_calc : temps en minutes (datetime_sortie - datetime_arrivee) --> temps passé aux urgences (total)

In [ ]:
df["_localisation_date_dentre_en_box_dt"].head()

0    2016-06-01 01:05:59.999997189
1    2016-06-01 01:32:00.000001020
2    2016-06-01 00:40:59.999997959
3    2016-06-01 01:36:00.000003061
4    2016-06-01 01:36:00.000003061
Name: _localisation_date_dentre_en_box_dt, dtype: object

In [ ]:
df["_informations_du_passage_date_dbut_prise_en_charge_mdicale_dt"].head()

0    2016-06-01 01:02:59.999999749
1    2016-06-01 02:13:59.999995407
2                              NaN
3    2016-06-01 01:57:00.000000250
4    2016-06-01 02:35:00.000000769
Name: _informations_du_passage_date_dbut_prise_en_charge_mdicale_dt, dtype: object

In [ ]:
def parse_date(date_str):
    return pd.to_datetime(date_str).normalize()

def parse_time(time_str):
    t = str(time_str).strip().lower().replace("h", ":").replace(" ", "")
    if ":" not in t:
        t = f"{t}:00"
    if len(t.split(":")[0]) == 1:
        t = "0" + t
    return pd.to_datetime(t, format="%H:%M").time()

def build_interval(date_str, start_time_str, end_time_str):
    d = parse_date(date_str)
    t0 = parse_time(start_time_str)
    t1 = parse_time(end_time_str)
    start_dt = pd.Timestamp.combine(d.date(), t0)
    end_dt = pd.Timestamp.combine(d.date(), t1)
    return start_dt, end_dt

def fifo_by_arrival_interval(df_raw, date_str, start_time_str, end_time_str):
    df = df_raw.copy()
    df["datetime_arrivee"] = pd.to_datetime(df["datetime_arrivee"], errors="coerce")
    df = df.dropna(subset=["datetime_arrivee"])

    start_dt, end_dt = build_interval(date_str, start_time_str, end_time_str)

    df = df[(df["datetime_arrivee"] >= start_dt) &
            (df["datetime_arrivee"] < end_dt)]

    df_sorted = df.sort_values("datetime_arrivee").reset_index(drop=True)

    return df_sorted[["num_dordre", "datetime_arrivee"]]

In [ ]:
# Exemple
df_fifo = fifo_by_arrival_interval(df, "2016-06-01", "10:00", "11:00")
print(df_fifo)

    num_dordre    datetime_arrivee
0           47 2016-06-01 10:00:00
1           48 2016-06-01 10:22:00
2           49 2016-06-01 10:23:00
3           50 2016-06-01 10:25:00
4           51 2016-06-01 10:30:00
5           52 2016-06-01 10:37:00
6           53 2016-06-01 10:43:00
7           54 2016-06-01 10:46:00
8           55 2016-06-01 10:51:00
9           56 2016-06-01 10:54:00
10          57 2016-06-01 10:55:00


In [ ]:
df[df["num_dordre"] == 47].head()

,num_dordre,date,heure,_informations_du_passage_date_et_heure_darrive,_informations_du_passage_moyen_darrive,_iao_motif_dentre,_iao_motif_de_venue,_patient_age_en_anne,_antcdent_antcdents_mdicaux,_antcdent_antcdents_chirurgicaux,...,is_retour_domicile,is_transfert,is_hospit,delai_arrivee_PEC_cap,delai_arrivee_box_cap,ind_complexite,ind_chronicite_delais,ind_issue,ind_congestion,ind_specialisation_examens
46,47,2016-06-01,10:00:00,2016-06-01 10:00:00,MOYENS PERSONNELS,0A CC _ AT du 30/05 lombosciatalgie après...,Rhumatologie non traumatique: Lombalgie,59,Apnée du sommeil appareillé RGO,hernie discal L5/S1 en 1996. nucléoplastie L4...,...,1,0,0,24.0,51.0,-0.717514,1.136942,0.734768,-1.449917,1.733312


### Gravité

In [ ]:
df._diagnostic_code_ccmu.value_counts()

_diagnostic_code_ccmu
2    229295
1     33293
3     19637
P      6068
4      2864
5      1738
D       227
Name: count, dtype: int64

P = psy
D = décès

In [ ]:
df[df._diagnostic_code_ccmu == "P"].head(20)

,num_dordre,date,heure,_informations_du_passage_date_et_heure_darrive,_informations_du_passage_moyen_darrive,_iao_motif_dentre,_iao_motif_de_venue,_patient_age_en_anne,_antcdent_antcdents_mdicaux,_antcdent_antcdents_chirurgicaux,...,is_retour_domicile,is_transfert,is_hospit,delai_arrivee_PEC_cap,delai_arrivee_box_cap,ind_complexite,ind_chronicite_delais,ind_issue,ind_congestion,ind_specialisation_examens
14,15,2016-06-01,04:16:00,2016-06-01 04:16:00,AMBULANCE PRIVEE,"retrouvé sur VP, patient bipolaire, demande un...",Psychiatrie: Bouffée délirante aigue/délire De...,30,"Patient schizophrène, DNID Notion d'inobserva...",NaN,...,1,0,0,18.0,NaN,-1.579155,-0.441408,0.610856,-1.597380,1.914429
132,133,2016-06-01,15:45:00,2016-06-01 15:45:00,AMBULANCE PRIVEE,sd depressif,Psychiatrie: Anxiété,46,NaN,NaN,...,1,0,0,109.0,NaN,-1.536265,-0.830006,0.062340,0.570880,0.784345
139,140,2016-06-01,16:24:00,2016-06-01 16:24:00,AMBULANCE PRIVEE,"agitation , agressivité","Psychiatrie: Agitation, agressivité",24,Asthme dans l'enfance plusieurs épisodes de p...,NaN,...,0,1,0,22.0,NaN,-0.248106,-0.012709,-2.121285,-0.225400,1.317599
415,416,2016-06-02,21:38:00,2016-06-02 21:38:00,MOYENS PERSONNELS,crise d'angoisse apres une mauvaise nouvelle,Psychiatrie: Anxiété,20,Asthme Crise d'angoisse,NaN,...,1,0,0,62.0,121.0,-1.493373,0.014885,0.523270,0.866309,1.177637
416,417,2016-06-02,21:52:00,2016-06-02 21:52:00,AMBULANCE PRIVEE,angoisse - 1er episode d'hallucinations auditi...,Psychiatrie: Anxiété,19,Aucun,NaN,...,0,1,0,69.0,54.0,0.686190,0.407078,-1.807844,0.481538,1.234125
425,426,2016-06-02,23:08:00,2016-06-02 23:08:00,MOYENS PERSONNELS,detresse psychologique,Psychiatrie: Anxiété,20,NaN,NaN,...,1,0,0,72.0,42.0,-1.923175,-1.193061,0.038533,1.107743,0.699015
439,440,2016-06-03,01:52:00,2016-06-03 01:52:00,AMBULANCE PRIVEE,mouvements incontrolés du MSD depuis hier,NaN,43,Deficience mentale légére,HTIC avec dérivation ventriculo-péritonéale s...,...,1,0,0,41.0,31.0,-0.956910,0.874101,0.974440,-2.175173,2.066613
548,549,2016-06-03,15:25:00,2016-06-03 15:25:00,MOYENS PERSONNELS,trble cmpt + hallucination visuelle transfer...,Psychiatrie: Demande de consultation psychiatr...,32,NaN,NaN,...,0,1,0,54.0,81.0,2.882009,-1.556371,-1.655355,0.809472,0.552993
584,585,2016-06-03,18:25:00,2016-06-03 18:25:00,AMBULANCE PRIVEE,crise de boulimie+ anorexie mentale,Divers: AEG - Altération de l'état généralPsyc...,25,NaN,NaN,...,1,0,0,45.0,31.0,-2.038682,-1.363723,0.092358,0.023217,1.270774
618,619,2016-06-03,20:10:00,2016-06-03 20:10:00,AMBULANCE PRIVEE,syndrôme depressif + idées sucidaires bipola...,Psychiatrie: Idées suicidaires,43,BPCO post tabagique non sevrée Schizophrénie ...,Appendicectomie Fracture de cheville gauche,...,0,1,0,35.0,NaN,0.254030,1.343701,-1.813029,-0.796890,1.915206


In [ ]:
df[df._diagnostic_code_ccmu == "D"]._iao_motif_de_venue.size

227

In [ ]:
df[df._diagnostic_code_ccmu == "D"]._iao_motif_de_venue.isna().sum()

np.int64(169)

In [ ]:
df[df._diagnostic_code_ccmu == "D"]._orientation_type_dorientation.value_counts()

_orientation_type_dorientation
DECES                221
TRANSFERT INTERNE      3
RETOUR DOMICILE        3
Name: count, dtype: int64

In [ ]:
df[df._diagnostic_code_ccmu == "D"]._iao_motif_de_venue.value_counts()

_iao_motif_de_venue
Pneumologie: Détresse respiratoire majeure                                                                                                 6
Pneumologie: Dyspnée sans détresse                                                                                                         6
Gastro-entérologie: Douleur abdominale (patient non valide/nécessitant brancard)                                                           4
Divers: AEG - Altération de l'état général                                                                                                 4
Neuro-chirurgie: Transfert SMUR pour prise en charge neurochir => Transfert Déchoc chir                                                    3
Traumatologie: Traumatisme nécessitant un transfert au déchoc chir (cf. critères de Vittel)                                                2
Neurologie: Céphalées                                                                                                                 

In [ ]:
def prepare_and_filter(df_raw, date_str, start_time_str, end_time_str):
    df = df_raw.copy()
    df["datetime_arrivee"] = pd.to_datetime(df["datetime_arrivee"], errors="coerce")
    df["_diagnostic_code_ccmu"] = df["_diagnostic_code_ccmu"].astype(str).str.strip()
    df = df.dropna(subset=["datetime_arrivee"])

    start_dt, end_dt = build_interval(date_str, start_time_str, end_time_str)
    df = df[(df["datetime_arrivee"] >= start_dt) & (df["datetime_arrivee"] < end_dt)].copy()

    df = df[df["_diagnostic_code_ccmu"].notna()].copy()

    order_map = {"D": 0, "5": 1, "4": 2, "3": 3, "2": 4, "1": 5, "P": 6}
    df["prio_rank"] = df["_diagnostic_code_ccmu"].map(order_map)
    df = df[df["prio_rank"].notna()].copy()

    return df.sort_values("datetime_arrivee").reset_index(drop=True)

def compute_order(df_arrived):
    df_sorted = df_arrived.sort_values(
        ["prio_rank", "datetime_arrivee"],
        ascending=[True, True]
    ).reset_index(drop=True)

    df_sorted["patient_num"] = df_sorted.index + 1
    return df_sorted[["patient_num", "datetime_arrivee", "_diagnostic_code_ccmu"]]

In [ ]:
# Exemple :
changes = simulate_reordering_changes(df, "2016-07-01", "20:00", "22:00")
changes


Changement d'ordonnancement à 2016-07-01 20:05:00
 patient_num    datetime_arrivee _diagnostic_code_ccmu
           1 2016-07-01 20:05:00                     P

Changement d'ordonnancement à 2016-07-01 20:07:00
 patient_num    datetime_arrivee _diagnostic_code_ccmu
           1 2016-07-01 20:07:00                     2
           2 2016-07-01 20:05:00                     P

Changement d'ordonnancement à 2016-07-01 20:24:00
 patient_num    datetime_arrivee _diagnostic_code_ccmu
           1 2016-07-01 20:07:00                     2
           2 2016-07-01 20:24:00                     2
           3 2016-07-01 20:05:00                     P

Changement d'ordonnancement à 2016-07-01 20:40:00
 patient_num    datetime_arrivee _diagnostic_code_ccmu
           1 2016-07-01 20:07:00                     2
           2 2016-07-01 20:24:00                     2
           3 2016-07-01 20:40:00                     2
           4 2016-07-01 20:05:00                     P

Changement d'ordonnanceme

[(Timestamp('2016-07-01 20:05:00'),
     patient_num    datetime_arrivee _diagnostic_code_ccmu
  0            1 2016-07-01 20:05:00                     P),
 (Timestamp('2016-07-01 20:07:00'),
     patient_num    datetime_arrivee _diagnostic_code_ccmu
  0            1 2016-07-01 20:07:00                     2
  1            2 2016-07-01 20:05:00                     P),
 (Timestamp('2016-07-01 20:24:00'),
     patient_num    datetime_arrivee _diagnostic_code_ccmu
  0            1 2016-07-01 20:07:00                     2
  1            2 2016-07-01 20:24:00                     2
  2            3 2016-07-01 20:05:00                     P),
 (Timestamp('2016-07-01 20:40:00'),
     patient_num    datetime_arrivee _diagnostic_code_ccmu
  0            1 2016-07-01 20:07:00                     2
  1            2 2016-07-01 20:24:00                     2
  2            3 2016-07-01 20:40:00                     2
  3            4 2016-07-01 20:05:00                     P),
 (Timestamp('2016-07-0

### Durée

In [ ]:
def parse_date(date_str):
    return pd.to_datetime(date_str).normalize()

def parse_time(time_str):
    t = str(time_str).strip().lower().replace("h", ":").replace(" ", "")
    if ":" not in t:
        t = f"{t}:00"
    if len(t.split(":")[0]) == 1:
        t = "0" + t
    return pd.to_datetime(t, format="%H:%M").time()

def build_interval(date_str, start_time_str, end_time_str):
    d = parse_date(date_str)
    t0 = parse_time(start_time_str)
    t1 = parse_time(end_time_str)
    start_dt = pd.Timestamp.combine(d.date(), t0)
    end_dt = pd.Timestamp.combine(d.date(), t1)
    return start_dt, end_dt

def prepare_and_filter_spt(df_raw, date_str, start_time_str, end_time_str):
    df = df_raw.copy()
    df["datetime_arrivee"] = pd.to_datetime(df["datetime_arrivee"], errors="coerce")
    df["delai_total_calc"] = pd.to_numeric(df["delai_total_calc"], errors="coerce")
    df = df.dropna(subset=["datetime_arrivee", "delai_total_calc"])

    start_dt, end_dt = build_interval(date_str, start_time_str, end_time_str)
    df = df[(df["datetime_arrivee"] >= start_dt) & (df["datetime_arrivee"] < end_dt)].copy()

    df = df[df["delai_total_calc"] > 0].copy()
    return df.sort_values("datetime_arrivee").reset_index(drop=True)

def compute_spt_order(df_arrived):
    df_sorted = df_arrived.sort_values(
        ["delai_total_calc", "datetime_arrivee"],
        ascending=[True, True]
    ).reset_index(drop=True)

    df_sorted["patient_num"] = df_sorted.index + 1
    return df_sorted[["patient_num", "datetime_arrivee", "delai_total_calc"]]

def simulate_spt_reordering_changes(df_raw, date_str, start_time_str, end_time_str):
    df = prepare_and_filter_spt(df_raw, date_str, start_time_str, end_time_str)
    if df.empty:
        return []

    changes = []
    prev_order = None

    arrival_times = df["datetime_arrivee"].sort_values().unique()

    for t in arrival_times:
        arrived = df[df["datetime_arrivee"] <= t].copy()
        order_df = compute_spt_order(arrived)

        current_order = list(zip(
            order_df["datetime_arrivee"].astype(str),
            order_df["delai_total_calc"].astype(float)
        ))

        if prev_order is None or current_order != prev_order:
            print(f"\nChangement d'ordonnancement à {pd.Timestamp(t)}")
            print(order_df.to_string(index=False))
            changes.append((pd.Timestamp(t), order_df))
            prev_order = current_order

    return changes

In [ ]:
# Exemple :
changes = simulate_spt_reordering_changes(df, "2016-06-01", "10:00", "11:00")
